# Real Data Predictions
Load training data → train model → preprocess REAL_DATA → predict → export

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Step 1 — Load & Inspect REAL_DATA

In [ ]:
df_real = pd.read_csv('../data/REAL_DATA.csv')
print('Shape:', df_real.shape)
print('Columns:', df_real.columns.tolist())
print('\nDtypes:\n', df_real.dtypes)
print('\nMissing values:\n', df_real.isna().sum())
df_real.head()

In [ ]:
# Inspect categorical and binary columns
print('state_holiday unique:', df_real['state_holiday'].unique())
print('open unique:         ', df_real['open'].unique())
print('Closed rows:         ', len(df_real[df_real['open'] == 0]))
print('Open rows:           ', len(df_real[df_real['open'] == 1]))

## Step 2 — Preprocess REAL_DATA

In [ ]:
# Drop index column — just a row identifier, not a feature
df_real = df_real.drop(columns=['index'])
print('Dropped index column. Shape:', df_real.shape)

In [ ]:
# Fix date column type: string → datetime → extract numeric features
# REAL_DATA format: DD/MM/YYYY — different from training (YYYY-MM-DD)
# dayfirst=True is critical — without it 01/03/2015 reads as January 3rd instead of March 1st
df_real['date']       = pd.to_datetime(df_real['date'], dayfirst=True)
df_real['year']       = df_real['date'].dt.year         # captures year-over-year trends
df_real['month']      = df_real['date'].dt.month        # captures seasonality (Dec = high sales)
df_real['day']        = df_real['date'].dt.day          # captures month-end effects
df_real['week']       = df_real['date'].dt.isocalendar().week.astype(int)  # weekly patterns
df_real['is_weekend'] = df_real['date'].dt.dayofweek.isin([5, 6]).astype(int)  # weekend flag
df_real = df_real.drop(columns=['date'])  # drop original string column — no longer needed
print('Date features extracted. Shape:', df_real.shape)

In [ ]:
# Encode state_holiday — One-Hot Encoding (required for Linear Regression)
# Values: '0'=no holiday, 'a'=public, 'b'=easter, 'c'=christmas
# drop_first=True: drops holiday_0 to avoid dummy variable trap (multicollinearity)
# Must cast to str BEFORE get_dummies to ensure consistent type
df_real['state_holiday'] = df_real['state_holiday'].astype(str)
df_real = pd.get_dummies(df_real, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int (0/1) for model compatibility
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_real.columns:
        df_real[col] = df_real[col].astype(int)

print('Encoding done. Columns:', df_real.columns.tolist())
df_real.head()

## Step 3 — Load & Preprocess TRAINING DATA (to train the model)

In [ ]:
df_train = pd.read_csv('../data/training.csv')
print('Training shape:', df_train.shape)

In [ ]:
# Drop index column — just a row identifier, not a feature
df_train = df_train.drop(columns=['Unnamed: 0'])

# Drop the 1 anomalous row: open store + customers > 0 + sales = 0
# Reason: 0.0002% of data — data entry error, contradicts the learned pattern
anomaly_mask = (df_train['open'] == 1) & (df_train['nb_customers_on_day'] > 0) & (df_train['sales'] == 0)
df_train = df_train[~anomaly_mask]
print(f'Removed {anomaly_mask.sum()} anomalous row(s)')

# Keep only open stores — closed stores always have sales=0
# They don't teach the model anything about what drives sales
# We will handle closed stores separately at prediction time (force sales=0)
df_train = df_train[df_train['open'] == 1].copy()
print(f'Rows after removing closed stores: {len(df_train)}')

In [ ]:
# Fix date column type: string → datetime → extract numeric features
# Training data format: YYYY-MM-DD
df_train['date']       = pd.to_datetime(df_train['date'])
df_train['year']       = df_train['date'].dt.year         # captures year-over-year trends
df_train['month']      = df_train['date'].dt.month        # captures seasonality (Dec = high sales)
df_train['day']        = df_train['date'].dt.day          # captures month-end effects
df_train['week']       = df_train['date'].dt.isocalendar().week.astype(int)  # weekly patterns
df_train['is_weekend'] = df_train['date'].dt.dayofweek.isin([5, 6]).astype(int)  # weekend flag
df_train = df_train.drop(columns=['date'])  # drop original string column — no longer needed

# Encode state_holiday — One-Hot Encoding (required for Linear Regression)
# Values: '0'=no holiday, 'a'=public, 'b'=easter, 'c'=christmas
# drop_first=True: drops holiday_0 to avoid dummy variable trap (multicollinearity)
# Must cast to str BEFORE get_dummies to ensure consistent type
df_train['state_holiday'] = df_train['state_holiday'].astype(str)
df_train = pd.get_dummies(df_train, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int (0/1) for model compatibility
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(int)

print('Training preprocessing done. Shape:', df_train.shape)
df_train.head()

## Step 4 — Define Features & Target

In [ ]:
# X = all features (everything except what we want to predict)
# y = target variable (what we want to predict)
X = df_train.drop(columns=['sales'])
y = df_train['sales']

# Save feature columns — needed to align REAL_DATA to same structure as training
feature_columns = X.columns.tolist()
print('Features:', feature_columns)
print('Target: sales')
print('X shape:', X.shape, '| y shape:', y.shape)

## Step 5 — Train / Validation Split

In [ ]:
# Split into training (75%) and validation (25%) sets
# random_state=42 ensures reproducibility (same split every run)
# Validation set simulates how the model performs on unseen data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Train size:      {len(X_train)}')
print(f'Validation size: {len(X_val)}')

## Step 6 — Train Linear Regression

In [ ]:
# Train the model — LinearRegression finds the best-fit line through the data
# It learns the weight (coefficient) of each feature
lr = LinearRegression()
lr.fit(X_train, y_train)
print('Model trained.')

## Step 7 — Evaluate

In [ ]:
# R2 Train      — how well the model learned the training data
# R2 Validation — how much we expect the model to perform on unseen data (our prediction)
# RMSE          — average error in euros (lower = better)
# R2 Difference — gap between train and validation (close to 0 = good generalisation)

y_train_pred = lr.predict(X_train)
y_val_pred   = lr.predict(X_val)

r2_train      = r2_score(y_train, y_train_pred)
r2_validation = r2_score(y_val,   y_val_pred)
rmse          = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_difference = r2_train - r2_validation

print('=== Linear Regression Results ===')
print(f'R2 Train      : {r2_train:.4f}  ← how well model learned training data')
print(f'R2 Validation : {r2_validation:.4f}  ← expected performance on real data')
print(f'RMSE          : {rmse:.2f}  ← average error in euros')
print(f'R2 Difference : {r2_difference:.4f}  ← gap (closer to 0 = better generalisation)')

## Step 8 — Predict on REAL_DATA

In [ ]:
# Align REAL_DATA columns to exactly match training features
# fill_value=0 handles any column present in training but missing in REAL_DATA
df_real_aligned = df_real.reindex(columns=feature_columns, fill_value=0)
print('Aligned shape:', df_real_aligned.shape)
print('Columns match:', df_real_aligned.columns.tolist() == feature_columns)

In [ ]:
# Predict open stores with the model
# Force closed stores to sales=0 (business rule: no customers = no sales)
predictions = np.zeros(len(df_real))

# Open stores: use model
open_mask = df_real['open'] == 1
predictions[open_mask] = lr.predict(df_real_aligned[open_mask])

# Closed stores: always 0 (business rule)
predictions[~open_mask] = 0

# Round to integers (sales are whole numbers) and clip negatives to 0
predictions = predictions.clip(min=0).round().astype(int)

print('Predictions done.')
print('Sample predictions:', predictions[:10])

## Step 9 — Export to CSV

In [ ]:
# Load original REAL_DATA to preserve the original format required by the teacher
df_output = pd.read_csv('../data/REAL_DATA.csv')
df_output['sales'] = predictions

# Export with the same format as the input file
df_output.to_csv('../data/group_4.csv', index=False)

print('Exported group_4.csv')
print('Shape:', df_output.shape)
df_output[['store_ID', 'date', 'open', 'sales']].head(10)

In [ ]:
# Sanity check: closed stores must have sales = 0
closed_with_sales = len(df_output[(df_output['open'] == 0) & (df_output['sales'] > 0)])
print('Closed stores with sales > 0:', closed_with_sales, '← must be 0')

# Summary of predictions for open stores
print('\nOpen stores prediction stats:')
print(df_output[df_output['open'] == 1]['sales'].describe())